Здесь я смотрю на баланс классов и разбиваю выборку на обучающую и отложенную.

In [1]:
SEED = 42


In [2]:
import polars as pl

full_dataset = pl.read_csv('../data/processed/dataset_clean_label.csv')


In [3]:
import json

error_type_ids = open('../data/processed/error_type_ids.json',
                      'r', -1, 'utf-8')
error_type_names = {v: k for k, v in json.load(error_type_ids).items()}
error_type_ids.close()


In [4]:
full_dataset\
    .group_by('error_type_label')\
    .len()\
    .with_columns(pl.col('error_type_label')\
                  .replace_strict(error_type_names, default=None)\
                  .alias('error_type_name'))\
    .sort('len', descending=True)

error_type_label,len,error_type_name
i64,u32,str
0,19775,"""нет"""
4,1153,"""задержка"""
3,414,"""дизартрия"""
7,339,"""фонетическая парафазия"""
6,185,"""семантическая парафазия"""
2,124,"""аномия"""
5,79,"""поиск слова"""
1,32,"""speech arrest"""


In [23]:
from sklearn.model_selection import train_test_split

dataset_train, dataset_test = train_test_split(full_dataset, 
                            test_size=0.25,
                            stratify=full_dataset['error_type_label'],
                            random_state=SEED)


In [6]:
dataset_train\
    .group_by('error_type_label')\
    .len()\
    .with_columns(pl.col('error_type_label')\
                  .replace_strict(error_type_names, default=None)\
                  .alias('error_type_name'))\
    .sort('len', descending=True)


error_type_label,len,error_type_name
i64,u32,str
0,14831,"""нет"""
4,865,"""задержка"""
3,310,"""дизартрия"""
7,254,"""фонетическая парафазия"""
6,139,"""семантическая парафазия"""
2,93,"""аномия"""
5,59,"""поиск слова"""
1,24,"""speech arrest"""


In [7]:
dataset_test\
    .group_by('error_type_label')\
    .len()\
    .with_columns(pl.col('error_type_label')\
                  .replace_strict(error_type_names, default=None)\
                  .alias('error_type_name'))\
    .sort('len', descending=True)


error_type_label,len,error_type_name
i64,u32,str
0,4944,"""нет"""
4,288,"""задержка"""
3,104,"""дизартрия"""
7,85,"""фонетическая парафазия"""
6,46,"""семантическая парафазия"""
2,31,"""аномия"""
5,20,"""поиск слова"""
1,8,"""speech arrest"""


In [45]:
dataset_train.write_csv('../data/processed/train.csv')
dataset_test.write_csv('../data/processed/test.csv')


С наличием транскрипций:

In [27]:
dataset_train\
    .filter(pl.col('^.+transcription.+$').str.len_chars() > 0)\
    ['error_type_label']\
    .value_counts()\
    .sort('error_type_label')\
    .with_columns(
        (pl.col('count') / pl.sum('count')).alias('share')
    )

error_type_label,count,share
i64,u32,f64
0,52,0.037627
2,39,0.02822
3,279,0.201881
4,646,0.467438
5,52,0.037627
6,71,0.051375
7,243,0.175832


In [29]:
dataset_test\
    .filter(pl.col('^.+transcription.+$').str.len_chars() > 0)\
    ['error_type_label']\
    .value_counts()\
    .sort('error_type_label')\
    .with_columns(
        (pl.col('count') / pl.sum('count')).alias('share')
    )

error_type_label,count,share
i64,u32,f64
0,11,0.023861
1,1,0.002169
2,12,0.02603
3,91,0.197397
4,223,0.483731
5,19,0.041215
6,23,0.049892
7,81,0.175705
